In [ ]:
# # sample number

# # j = '_arcsinh'+'18.2'+'a'# '18.2'+'a'#
# # j = '18'+'a'#  '_arcsinh'+'18.2'+'a'#'14.2'+'a'#'_raw'+'14.2'+'a'#
            
# # '7.1a','13a','14a','15a','17a','18a','19a','20a','8.1a'

# # j = '13'+'a'
# # j='b2345'#all samples

# # j = '4'
# # j = 'b2345'
# # samples = []


# # pdx
# # j = 'b2'
# # samples = ['5.2','6.2','7.2','8.2']
# j = 'b3'
# # samples = ['1.3']
# samples = ['1.3','2.3','3.3','4.3','5.3','6.3',]
# # samples = ['1.3','2.3','3.3','4.3','5.3','6.3',]

# # j = '19a'
# # j = '18.2'+'a'
# # j = 'b2345'+'s'#stroma - exists only in batch

# # j = 'b2345'+'a'
# # j = 'b2345'+'s'+'_prescaled'
# # j = 'b5_not_fitted'

# args = {
#         'j' : j,
#         'samples' :samples,
#         'feautures_ind':2,
#         # 'data_folder' : 'Data_',
#         # 'data_folder' : 'age_prescaled',
#         # create visualization (umap, dbscan are always shown)
#         # 'visualize':True,'plotUMAP' : True,
        
#         # show figures in notebook
#         'show' : True, 'format' :'pdf',
#         'recalculate_umap' : True,# calculate umap
#         'recalculate_db' : True,# calculate dbscan
#         'create_adjusted' : False,
#         'compare' : False,# compare 11,stroma and rest of the cells
#         'stroma': True,#with or without stroma in data
#         'normalized' : True,#data with fixed batch effect
#         'predict_noise': False,
#         'pdx': True,
#         }
# # create_adjusted = False
# # compare = False# compare 11,stroma and rest of the cells
# # print(j, 'ind:',config['feautures_ind'], dir_data1, config, create_adjusted)

# # parent_dir = '/home/yishai/breast_cancer_PHD_research'


# use_MASTER = True

# %run imports.ipynb # import all imported packages from imports.ipynb
# samples = Series(df['samp'])
# uniq_samples = samples.unique()# don't drop anchor samples

# uniq_samples
# len(df)
# df.to_parquet(f'{config["dir_plots"]}/df.parquet')





In [ ]:
import anndata as ad
import numpy as np
import pandas as pd




# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.metrics import (
#     roc_auc_score,
#     roc_curve,
#     precision_score,
#     recall_score,
#     confusion_matrix,
#     d2_log_loss_score,
#     cohen_kappa_score
# )
# from sklearn.model_selection import StratifiedKFold
# from scipy.spatial import distance, ConvexHull
from scipy.stats import norm
from scipy import sparse
# import umap
# import torch
# from math import comb
from typing import Optional, Dict, Tuple, List
from collections.abc import Mapping
import tqdm

In [ ]:
# ---------------------------------------------------------------------
# Permutation-based Cell Scoring (formerly sipsic_like_scores_v3)
# ---------------------------------------------------------------------

def _to_weight_pairs(obj):
    if obj is None: return []
    if isinstance(obj, Mapping) and set(obj.keys()) & {"up", "down"}:
        pairs = []
        for g in (obj.get("up", []) or []):   pairs.append((g, +1.0))
        for g in (obj.get("down", []) or []): pairs.append((g, -1.0))
        return pairs
    if isinstance(obj, Mapping):
        return [(k, float(v)) for k, v in obj.items()]
    if isinstance(obj, (list, tuple)) and len(obj) > 0 and isinstance(obj[0], (list, tuple)) and len(obj[0]) == 2:
        return [(str(k), float(v)) for (k, v) in obj]
    if isinstance(obj, (set, list, tuple)):
        return [(str(k), 1.0) for k in obj]
    if isinstance(obj, str):
        return [(obj, 1.0)]
    raise TypeError(f"Unsupported marker-set format: {type(obj)} -> {obj}")

def normalize_weighted_marker_sets(
    var_names: np.ndarray,
    marker_sets: Dict[str, object],
    tolerate_prefixes: bool = True,
    normalize_set_weights: Optional[str] = None,
    use_sparse_W: bool = False
):
    G = len(var_names)
    vlow = np.char.lower(var_names.astype(str))
    set_names, sizes = [], []

    if use_sparse_W:
        data, indices, indptr = [], [], [0]
    rows_dense = []

    for sname, payload in marker_sets.items():
        pairs = _to_weight_pairs(payload)
        hits_idx, hits_w = [], []
        for g, wt in pairs:
            g_low = str(g).lower()
            hit = (vlow == g_low)
            if tolerate_prefixes:
                hit |= np.char.startswith(vlow, g_low + "(") | np.char.startswith(vlow, g_low + "-")
            idx = np.where(hit)[0]
            if idx.size:
                hits_idx.extend(idx.tolist())
                hits_w.extend([float(wt)] * idx.size)

        if not hits_idx: continue

        if len(hits_idx) != len(set(hits_idx)):
            df = pd.DataFrame({"i": hits_idx, "w": hits_w})
            agg = df.groupby("i", sort=False)["w"].sum()
            hits_idx, hits_w = agg.index.to_numpy(dtype=int), agg.to_numpy(dtype=float)

        if normalize_set_weights in {"l2", "l1"}:
            denom = np.linalg.norm(hits_w) if normalize_set_weights == "l2" else np.sum(np.abs(hits_w))
            denom = float(denom) if denom != 0.0 else 1.0
            hits_w = (np.asarray(hits_w, dtype=np.float64) / denom).tolist()

        set_names.append(sname)
        sizes.append(len(hits_idx))

        if use_sparse_W:
            indices.extend(hits_idx)
            data.extend(hits_w)
            indptr.append(len(indices))
        else:
            w = np.zeros(G, dtype=np.float32)
            w[np.asarray(hits_idx, dtype=int)] = np.asarray(hits_w, dtype=np.float32)
            rows_dense.append(w)

    if not set_names: raise ValueError("No marker sets overlap the features.")

    sizes = np.asarray(sizes, dtype=int)
    if use_sparse_W:
        W = sparse.csr_matrix((np.asarray(data, dtype=np.float32),
                               np.asarray(indices, dtype=np.int32),
                               np.asarray(indptr, dtype=np.int32)),
                               shape=(len(set_names), G), dtype=np.float32)
    else:
        W = np.vstack(rows_dense).astype(np.float32)

    return set_names, W, sizes

def get_layer_matrix(adata, layer: str):
    X = adata.layers[layer] if (layer and layer in adata.layers) else adata.X
    return np.asarray(X, dtype=np.float32)

def perm_cell(
    adata,
    marker_sets: Dict[str, object],
    layer: str = "scaled",
    n_perm: int = 2000,
    seed: int = 0,
    exclude_set: bool = True,
    two_sided: bool = False,
    abs_variant: bool = True,
    exact_max_combinations: int = 50_000,
    progress: bool = True,
    normalize_set_weights: Optional[str] = None,
    use_sparse_W: bool = False,
    prefer_permutation: bool = True,
    perm_batch: int = 1024,
):
    """
    Compute permutation-based scores for marker sets relative to random background.
    Previously known as sipsic_like_scores_v3.
    """
    rng = np.random.default_rng(seed)
    X = get_layer_matrix(adata, layer).astype(np.float32)
    genes = np.asarray(adata.var_names)

    set_names, W, sizes = normalize_weighted_marker_sets(
        genes, marker_sets, tolerate_prefixes=True,
        normalize_set_weights=normalize_set_weights, use_sparse_W=use_sparse_W
    )
    N, G = X.shape
    S = len(set_names)

    Xc = X - X.mean(axis=1, keepdims=True)
    Xc_abs = np.abs(Xc) if abs_variant else None

    if use_sparse_W:
        obs = Xc @ W.T
        obs_abs = (Xc_abs @ (abs(W)).T) if abs_variant else None
    else:
        WT = W.T
        obs = Xc @ WT
        obs_abs = (Xc_abs @ np.abs(WT)) if abs_variant else None

    Z = np.zeros((N, S), np.float32)
    Pmat = np.ones((N, S), np.float64)
    Z_abs, P_abs = (np.zeros((N, S), np.float32), np.ones((N, S), np.float64)) if abs_variant else (None, None)

    def p_from_Z(z): return (2.0 * norm.sf(np.abs(z))) if two_sided else norm.sf(z)

    def update_stream(mu, m2, n_seen, batch):
        B = batch.shape[1]
        if B == 0: return mu, m2, n_seen
        bmean = batch.mean(axis=1)
        bm2 = ((batch - bmean[:, None])**2).sum(axis=1)
        new_n = n_seen + B
        delta = bmean - mu
        new_mu = mu + delta * (B / new_n)
        new_m2 = m2 + bm2 + (delta**2) * n_seen * B / new_n
        return new_mu, new_m2, new_n

    it = range(S)
    if progress: from tqdm import tqdm; it = tqdm(it, desc="Scoring weighted sets")

    if use_sparse_W:
        W_csr = W
        indptr, indices, dataW = W_csr.indptr, W_csr.indices, W_csr.data

    for s in it:
        m = int(sizes[s])
        
        if use_sparse_W:
            start, end = indptr[s], indptr[s+1]
            feat_idx = indices[start:end]
            w_nonzero = dataW[start:end].astype(np.float32)
            mask = np.zeros(G, dtype=bool); mask[feat_idx] = True
        else:
            w_row = W[s]
            mask = (w_row != 0)
            feat_idx = np.where(mask)[0]
            w_nonzero = w_row[feat_idx].astype(np.float32)

        pool = np.where(~mask)[0] if exclude_set else np.arange(G, dtype=int)
        if exclude_set and len(pool) < m: pool = np.arange(G, dtype=int)
        n_pool = len(pool)

        do_enum = False
        if not prefer_permutation and 0 <= m <= n_pool and m <= 50:
            try:
                if comb(n_pool, m) <= exact_max_combinations: do_enum = True
            except OverflowError: pass

        mu, m2, n_seen = np.zeros(N), np.zeros(N), 0
        mu_a, m2_a, n_seen_a = (np.zeros(N), np.zeros(N), 0) if abs_variant else (None, None, None)

        if do_enum:
            import itertools
            itc = itertools.combinations(pool, m)
            while True:
                batch = list(itertools.islice(itc, max(256, min(4096, perm_batch))))
                if not batch: break
                B = len(batch)
                idxs = np.asarray(batch, dtype=int)
                rows, cols = idxs.ravel(), np.repeat(np.arange(B), m)
                data = np.tile(w_nonzero, B)
                
                U = sparse.csr_matrix((data, (rows, cols)), shape=(G, B), dtype=np.float32)
                mu, m2, n_seen = update_stream(mu, m2, n_seen, Xc @ U)
                if abs_variant:
                    Ua = sparse.csr_matrix((np.abs(data), (rows, cols)), shape=(G, B), dtype=np.float32)
                    mu_a, m2_a, n_seen_a = update_stream(mu_a, m2_a, n_seen_a, Xc_abs @ Ua)
        else:
            nP = int(max(1, n_perm))
            b = 0
            while b < nP:
                B = min(perm_batch, nP - b)
                idxs = np.stack([rng.choice(pool, size=m, replace=False) for _ in range(B)], axis=0)
                rows, cols = idxs.ravel(), np.repeat(np.arange(B), m)
                data = np.tile(w_nonzero, B)
                
                U = sparse.csr_matrix((data, (rows, cols)), shape=(G, B), dtype=np.float32)
                mu, m2, n_seen = update_stream(mu, m2, n_seen, Xc @ U)
                if abs_variant:
                    Ua = sparse.csr_matrix((np.abs(data), (rows, cols)), shape=(G, B), dtype=np.float32)
                    mu_a, m2_a, n_seen_a = update_stream(mu_a, m2_a, n_seen_a, Xc_abs @ Ua)
                b += B

        sd = (np.sqrt(m2 / max(n_seen, 1)) + 1e-6)
        z = (obs[:, s] - mu) / sd
        Z[:, s] = z.astype(np.float32)
        Pmat[:, s] = p_from_Z(z)
        
        if abs_variant:
            sd_a = (np.sqrt(m2_a / max(n_seen_a, 1)) + 1e-6)
            z_a = (obs_abs[:, s] - mu_a) / sd_a
            Z_abs[:, s] = z_a.astype(np.float32)
            P_abs[:, s] = p_from_Z(z_a)

    Z_dir = (np.sign(obs) * (Z_abs if abs_variant else np.abs(Z))).astype(np.float32)
    Z_df, P_df, Zdir_df = pd.DataFrame(Z, index=adata.obs_names, columns=set_names), \
                          pd.DataFrame(Pmat, index=adata.obs_names, columns=set_names), \
                          pd.DataFrame(Z_dir, index=adata.obs_names, columns=set_names)

    if abs_variant:
        return Z_df, P_df, pd.DataFrame(Z_abs, index=adata.obs_names, columns=set_names), \
               pd.DataFrame(P_abs, index=adata.obs_names, columns=set_names), Zdir_df
    return Z_df, P_df, Zdir_df


In [ ]:

# # -----------------------------------
# # interactive labeling
# # --------------------------------

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
# import matplotlib.patches as mpatches
# from matplotlib.widgets import LassoSelector
# from matplotlib.path import Path
# from matplotlib.patches import Polygon
# from matplotlib.colors import ListedColormap, BoundaryNorm
# import seaborn as sns
# import ipywidgets as widgets
# from IPython.display import display, clear_output
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import plotly.express as px
# from typing import Optional, List, Dict, Tuple
# import warnings
# import random

# class InteractiveClusterLabeler:
#     """
#     Interactive cluster labeling using Plotly lasso selection on UMAP space.
#     """

#     def __init__(self, adata, umap_key: str = 'X_umap',
#                  features: Optional[List[str]] = None,
#                  subsample: Optional[int] = None):
#         """
#         Initialize the interactive cluster labeler.

#         Parameters
#         ----------
#         adata : AnnData
#             Annotated data matrix with UMAP coordinates in obsm
#         umap_key : str
#             Key in adata.obsm for UMAP coordinates (default: 'X_umap')
#         features : list of str, optional
#             List of features (markers) to make available for visualization.
#             If None, uses all features in adata.var_names
#         subsample : int, optional
#             If provided, subsample to this many cells for faster interaction
#         """
#         self.adata_original = adata
#         self.umap_key = umap_key

#         # Subsample if requested
#         if subsample is not None and subsample < adata.n_obs:
#             print(f"Subsampling {subsample} cells from {adata.n_obs}")
#             self.subsample_indices = np.random.choice(
#                 adata.n_obs, size=subsample, replace=False
#             )
#             self.adata = adata[self.subsample_indices].copy()
#         else:
#             self.adata = adata
#             self.subsample_indices = None

#         # Extract UMAP coordinates
#         if umap_key not in self.adata.obsm:
#             raise ValueError(f"UMAP key '{umap_key}' not found in adata.obsm")
#         self.umap = self.adata.obsm[umap_key]

#         # Set up features
#         if features is None:
#             self.features = list(self.adata.var_names)
#         else:
#             # Validate features exist
#             missing = [f for f in features if f not in self.adata.var_names]
#             if missing:
#                 raise ValueError(f"Features not found: {missing}")
#             self.features = features

#         # Initialize cluster labels (-1 = unlabeled)
#         self.cluster_labels = -np.ones(len(self.adata), dtype=int)
#         self.cluster_names: Dict[int, str] = {}  # cluster_id -> name
#         self.next_cluster_id = 0

#         # XGBoost model and predictions
#         self.classifier = None
#         self.predicted_labels = None
#         self.predicted_proba = None
#         self.prob_threshold = 0.5

#         # Available colormaps for feature visualization (Plotly colorscale names)
#         self.available_colormaps = [
#             'RdBu','viridis', 'plasma', 'inferno', 'magma', 'cividis',
#             'blues', 'greens', 'reds', 'purples', 'oranges',
#             'ylorrd', 'ylgnbu', 'turbo', 'hot', 'jet', 'rainbow',
#             'rdbu', 'rdylbu', 'rdylgn', 'spectral', 'piyg', 'brbg',
#             'puor', 'prgn', 'picnic', 'portland', 'earth',
#             'icefire', 'balance', 'curl', 'delta', 'tealrose'
            
#         ]
#         self.current_colormap = 'RdBu'

#         # UI state
#         self.current_feature = self.features[0] if self.features else None
#         self._selected_indices = []
#         self._fig_widget = None
#         self._cluster_fig_widget = None

#     def _get_feature_values(self, feature: str) -> np.ndarray:
#         """Extract feature values from adata."""
#         if feature not in self.adata.var_names:
#             raise ValueError(f"Feature '{feature}' not found")

#         F = self.adata[:, feature].X
#         if hasattr(F, 'toarray'):
#             F = F.toarray().flatten()
#         else:
#             F = np.asarray(F).flatten()
#         return F

#     def _get_colorbar_range(self, values: np.ndarray) -> Tuple[float, float, float]:
#         """
#         Get colorbar range with robust percentile scaling, centered at zero.
#         """
#         # p01 = np.nanpercentile(values, 1)
#         # p99 = np.nanpercentile(values, 99)
#         # abs_max = max(abs(p01), abs(p99))
#         # vmin, vmax = -abs_max, abs_max
#         vmid = 0.0
#         intensity = pd.Series(values)
#         vmax=intensity.quantile(0.99);vmin=intensity.quantile(0.01)
#         return vmin, vmid, vmax

#     def _create_feature_scatter(self, feature: str) -> go.FigureWidget:
#         """Create a scatter plot colored by feature values."""
#         values = self._get_feature_values(feature)
        
#         vmin, vmid, vmax = self._get_colorbar_range(values)

#         fig = go.FigureWidget()
#         fig.add_trace(go.Scatter(
#             x=self.umap[:, 0],
#             y=self.umap[:, 1],
#             mode='markers',
#             marker=dict(
#                 size=4,

#                 color=values,
#                 colorscale=self.current_colormap,
#                 cmin=vmin,
#                 # cmid=vmid,
#                 cmax=vmax,
#                 showscale=True,
#                 colorbar=dict(title=feature, x=1.02),
#             ),

#             text=[f"Cell {i}<br>{feature}: {values[i]:.2f}"
#                   for i in range(len(self.umap))],
#             hoverinfo='text',
#             name=feature,
#             selectedpoints=[],
#         ))
#         # print(vmin,vmid, vmax)

#         fig.update_layout(
#             title=f"UMAP colored by {feature} - Use Lasso to Select",
#             xaxis_title="UMAP 1",
#             yaxis_title="UMAP 2",
#             dragmode='lasso',
#             height=500,
#             width=600,
#             showlegend=False,
#         )
#         return fig

#     def _create_cluster_scatter(self) -> go.FigureWidget:
#         """Create a scatter plot colored by cluster assignments with legend."""
#         fig = go.FigureWidget()
#         colors = px.colors.qualitative.Plotly
#         self._add_cluster_traces(fig, colors)
#         fig.update_layout(
#             title="UMAP colored by Cluster Labels",
#             xaxis_title="UMAP 1",
#             yaxis_title="UMAP 2",
#             height=500,
#             width=650,
#             showlegend=True,
#             legend=dict(
#                 yanchor="top",
#                 y=0.99,
#                 xanchor="left",
#                 x=1.02,
#                 bgcolor="rgba(255,255,255,0.8)",
#             ),
#         )
#         return fig

#     def _add_cluster_traces(self, fig: go.FigureWidget, colors: list):
#         """Add scatter traces for each cluster to the figure."""
#         unlabeled_mask = self.cluster_labels == -1
#         if unlabeled_mask.any():
#             fig.add_trace(go.Scatter(
#                 x=self.umap[unlabeled_mask, 0],
#                 y=self.umap[unlabeled_mask, 1],
#                 mode='markers',
#                 marker=dict(size=4, color='lightgray', opacity=0.5),
#                 text=[f"Cell {i}: Unlabeled" for i in np.where(unlabeled_mask)[0]],
#                 hoverinfo='text',
#                 name='Unlabeled',
#                 showlegend=True,
#             ))

#         for cluster_id in sorted(self.cluster_names.keys()):
#             mask = self.cluster_labels == cluster_id
#             if not mask.any():
#                 continue

#             color = colors[cluster_id % len(colors)]
#             name = self.cluster_names.get(cluster_id, f"Cluster {cluster_id}")

#             fig.add_trace(go.Scatter(
#                 x=self.umap[mask, 0],
#                 y=self.umap[mask, 1],
#                 mode='markers',
#                 marker=dict(size=4, color=color, opacity=0.7),
#                 text=[f"Cell {i}: {name}" for i in np.where(mask)[0]],
#                 hoverinfo='text',
#                 name=name,
#                 showlegend=True,
#             ))

#     def _update_cluster_scatter(self):
#         """Update the cluster scatter plot with current labels."""
#         if self._cluster_fig_widget is None:
#             return
#         colors = px.colors.qualitative.Plotly
#         self._cluster_fig_widget.data = []
#         self._add_cluster_traces(self._cluster_fig_widget, colors)

#     def _on_selection(self, trace, points, selector):
#         """Handle lasso selection events."""
#         self._selected_indices = list(points.point_inds)
#         n_selected = len(self._selected_indices)
#         if n_selected > 0:
#             print(f"Selected {n_selected} points")

#     def label_points(self, indices: List[int], cluster_name: Optional[str] = None) -> int:
#         """Programmatically label a set of points as a new cluster."""
#         if not indices:
#             print("No indices provided")
#             return -1

#         if cluster_name is None:
#             cluster_name = f"Cluster {self.next_cluster_id}"

#         cluster_id = self.next_cluster_id

#         indices = [i for i in indices if 0 <= i < len(self.cluster_labels)]
#         n_labeled = 0
#         for idx in indices:
#             if self.cluster_labels[idx] == -1:
#                 self.cluster_labels[idx] = cluster_id
#                 n_labeled += 1

#         if n_labeled > 0:
#             self.cluster_names[cluster_id] = cluster_name
#             self.next_cluster_id += 1
#             print(f"Labeled {n_labeled} points as '{cluster_name}' (ID: {cluster_id})")
#             self._update_cluster_scatter()
#             self._update_summary()
#             self._update_rename_dropdown()
#             return cluster_id
#         else:
#             print("No new points labeled")
#             return -1

#     def _label_selection(self, button):
#         """Label currently selected points with a new cluster."""
#         if not self._selected_indices:
#             print("No points selected.")
#             return

#         cluster_name = self._cluster_name_input.value.strip() or f"Cluster {self.next_cluster_id}"
        
#         n_labeled = 0
#         cluster_id = self.next_cluster_id
#         for idx in self._selected_indices:
#             if self.cluster_labels[idx] == -1:
#                 self.cluster_labels[idx] = cluster_id
#                 n_labeled += 1
        
#         if n_labeled > 0:
#             self.cluster_names[cluster_id] = cluster_name
#             self.next_cluster_id += 1
#             print(f"Labeled {n_labeled} points as '{cluster_name}'")
#             self._update_cluster_scatter()
#             self._update_summary()
#             self._update_rename_dropdown()
#             self._selected_indices = []
#             self._cluster_name_input.value = ""
#         else:
#             print("No new points labeled")

#     def _clear_selection(self, button):
#         self._selected_indices = []
#         print("Selection cleared")

#     def _on_feature_change(self, change):
#         if change['name'] != 'value': return
#         feature = change['new']
#         self.current_feature = feature
#         values = self._get_feature_values(feature)
#         vmin, vmid, vmax = self._get_colorbar_range(values)
#         with self._fig_widget.batch_update():
#             self._fig_widget.data[0].marker.color = values
#             self._fig_widget.data[0].marker.cmin = vmin
#             self._fig_widget.data[0].marker.cmid = vmid
#             self._fig_widget.data[0].marker.cmax = vmax
#             self._fig_widget.data[0].marker.colorbar.title = feature
#             self._fig_widget.data[0].text = [f"Cell {i}<br>{feature}: {values[i]:.2f}" for i in range(len(self.umap))]
#             self._fig_widget.layout.title = f"UMAP colored by {feature} - Use Lasso to Select"

#     def _on_colormap_change(self, change):
#         if change['name'] != 'value': return
#         colormap = change['new']
#         self.current_colormap = colormap
#         if self._fig_widget is not None:
#             with self._fig_widget.batch_update():
#                 self._fig_widget.data[0].marker.colorscale = colormap

#     def _on_rename_cluster(self, button):
#         cluster_id = self._rename_cluster_dropdown.value
#         new_name = self._rename_input.value.strip()
#         if cluster_id is None or not new_name: return
#         self.cluster_names[cluster_id] = new_name
#         self._rename_input.value = ""
#         self._update_cluster_scatter()
#         self._update_summary()
#         self._update_rename_dropdown()

#     def _update_rename_dropdown(self):
#         if not hasattr(self, '_rename_cluster_dropdown'): return
#         options = [(f"{self.cluster_names.get(cid, f'Cluster {cid}')} (ID: {cid})", cid)
#                    for cid in sorted(self.cluster_names.keys())]
#         self._rename_cluster_dropdown.options = options if options else [('No clusters', None)]

#     def _on_remove_cluster(self, button):
#         cluster_id = self._rename_cluster_dropdown.value
#         if cluster_id is None: return
#         self.cluster_labels[self.cluster_labels == cluster_id] = -1
#         del self.cluster_names[cluster_id]
#         self._update_cluster_scatter()
#         self._update_summary()
#         self._update_rename_dropdown()

#     def _on_reorder_ids(self, button):
#         if not self.cluster_names: return
#         old_ids = sorted(self.cluster_names.keys())
#         id_mapping = {old_id: new_id for new_id, old_id in enumerate(old_ids)}
#         new_labels = self.cluster_labels.copy()
#         for old_id, new_id in id_mapping.items():
#             new_labels[self.cluster_labels == old_id] = new_id
#         self.cluster_labels = new_labels
#         self.cluster_names = {id_mapping[old_id]: name for old_id, name in self.cluster_names.items()}
#         self.next_cluster_id = len(self.cluster_names)
#         self._update_cluster_scatter()
#         self._update_summary()
#         self._update_rename_dropdown()

#     def _train_classifier(self, button):
#         try:
#             import xgboost as xgb
#         except ImportError:
#             print("XGBoost not installed. Run: pip install xgboost")
#             return

#         labeled_mask = self.cluster_labels >= 0
#         if labeled_mask.sum() == 0:
#             print("No points labeled yet.")
#             return

#         unique_labels = np.unique(self.cluster_labels[labeled_mask])
#         if len(unique_labels) < 2:
#             print("Need at least 2 different clusters.")
#             return

#         print(f"Training XGBoost classifier on {labeled_mask.sum()} labeled points...")
#         X = self.adata.X
#         if hasattr(X, 'toarray'): X = X.toarray()
#         X_train = X[labeled_mask]
#         y_train = self.cluster_labels[labeled_mask]

#         self.classifier = xgb.XGBClassifier(
#             n_estimators=100, max_depth=6, learning_rate=0.1,
#             objective='multi:softprob', num_class=len(unique_labels),
#             random_state=42, verbosity=0,
#         )

#         label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
#         idx_to_label = {idx: label for label, idx in label_to_idx.items()}
#         y_train_mapped = np.array([label_to_idx[y] for y in y_train])

#         self.classifier.fit(X_train, y_train_mapped)
        
#         # Predict on the FULL dataset
#         X_full = self.adata_original.X
#         if hasattr(X_full, 'toarray'): X_full = X_full.toarray()
        
#         proba_full = self.classifier.predict_proba(X_full)
#         pred_idx_full = np.argmax(proba_full, axis=1)
        
#         self.full_predicted_labels = np.array([idx_to_label[idx] for idx in pred_idx_full])
#         self.full_predicted_proba = np.max(proba_full, axis=1)

#         # If subsampled, map back to subsample for UI
#         if self.subsample_indices is not None:
#             self.predicted_labels = self.full_predicted_labels[self.subsample_indices]
#             self.predicted_proba = self.full_predicted_proba[self.subsample_indices]
#         else:
#             self.predicted_labels = self.full_predicted_labels
#             self.predicted_proba = self.full_predicted_proba

#         print(f"Classifier trained successfully! Predictions generated for {len(self.adata_original)} cells.")
#         self._apply_threshold(None)

#     def _apply_threshold(self, change):
#         if self.predicted_labels is None: return
#         threshold = self._threshold_slider.value if hasattr(self, '_threshold_slider') else self.prob_threshold
#         self.prob_threshold = threshold

#         final_labels = self.cluster_labels.copy()
#         unlabeled_mask = self.cluster_labels == -1
#         confident_mask = self.predicted_proba >= threshold
        
#         predicted_confident = unlabeled_mask & confident_mask
#         final_labels[predicted_confident] = self.predicted_labels[predicted_confident]
        
#         predicted_uncertain = unlabeled_mask & ~confident_mask
#         final_labels[predicted_uncertain] = -2
        
#         self._final_labels = final_labels
#         self._update_final_cluster_plot()

#     def _update_final_cluster_plot(self):
#         if self._cluster_fig_widget is None or not hasattr(self, '_final_labels'): return
#         colors = px.colors.qualitative.Plotly
#         self._cluster_fig_widget.data = []

#         na_mask = self._final_labels == -2
#         if na_mask.any():
#             self._cluster_fig_widget.add_trace(go.Scatter(
#                 x=self.umap[na_mask, 0], y=self.umap[na_mask, 1],
#                 mode='markers', marker=dict(size=4, color='black', opacity=0.7),
#                 text=[f"Cell {i}: NA (prob={self.predicted_proba[i]:.2f})" for i in np.where(na_mask)[0]],
#                 hoverinfo='text', name='NA (uncertain)', showlegend=True,
#             ))
            
#         unlabeled_mask = self._final_labels == -1
#         if unlabeled_mask.any():
#             self._cluster_fig_widget.add_trace(go.Scatter(
#                 x=self.umap[unlabeled_mask, 0], y=self.umap[unlabeled_mask, 1],
#                 mode='markers', marker=dict(size=4, color='lightgray', opacity=0.5),
#                 name='Unlabeled', showlegend=True,
#             ))

#         for cluster_id in sorted(self.cluster_names.keys()):
#             mask = self._final_labels == cluster_id
#             if not mask.any(): continue
#             color = colors[cluster_id % len(colors)]
#             name = self.cluster_names.get(cluster_id, f"Cluster {cluster_id}")
#             hover_texts = []
#             for i in np.where(mask)[0]:
#                 if self.cluster_labels[i] >= 0:
#                     hover_texts.append(f"Cell {i}: {name} (manual)")
#                 else:
#                     hover_texts.append(f"Cell {i}: {name} (pred, p={self.predicted_proba[i]:.2f})")
            
#             self._cluster_fig_widget.add_trace(go.Scatter(
#                 x=self.umap[mask, 0], y=self.umap[mask, 1],
#                 mode='markers', marker=dict(size=4, color=color, opacity=0.7),
#                 text=hover_texts, hoverinfo='text', name=name, showlegend=True,
#             ))

#     def save_full_labels(self, obs_key: str = 'predicted_labels', prob_key: str = 'predicted_proba'):
#         """
#         Save the predicted labels and probabilities for the FULL dataset to adata_original.obs.
        
#         Parameters
#         ----------
#         obs_key : str
#             Key to store the predicted labels in adata_original.obs
#         prob_key : str
#             Key to store the predicted probabilities in adata_original.obs
#         """
#         if not hasattr(self, 'full_predicted_labels'):
#             print("No classifier trained yet. Train a classifier first.")
#             return

#         # Apply threshold to full predictions
#         final_full_labels = self.full_predicted_labels.copy().astype(object)
#         uncertain_mask = self.full_predicted_proba < self.prob_threshold
#         final_full_labels[uncertain_mask] = 'Uncertain' # Or keep as string

#         self.adata_original.obs[obs_key] = final_full_labels
#         self.adata_original.obs[prob_key] = self.full_predicted_proba
#         print(f"Saved full predictions to adata.obs['{obs_key}'] and adata.obs['{prob_key}']")


#     def _update_summary(self):
#         n_labeled = (self.cluster_labels >= 0).sum()
#         n_unlabeled = (self.cluster_labels == -1).sum()
#         n_clusters = len(self.cluster_names)
#         summary = f"Labeled: {n_labeled} | Unlabeled: {n_unlabeled} | Clusters: {n_clusters}"
#         if hasattr(self, '_summary_text'):
#             self._summary_text.value = summary

#     def _finalize_assignments(self, button):
#         """Finalize the current assignments based on threshold and save to adata."""
#         if not hasattr(self, 'full_predicted_labels'):
#             print("No classifier trained yet.")
#             return
            
#         self.save_full_labels()
#         self._update_summary() # Update summary to potentially reflect saved state if we decide to track that
#         # Provide visual feedback
#         button.description = "Saved!"
#         button.icon = "check-circle" 
#         import time
#         # Note: In a real async UI this simple sleep might block, 
#         # but for ipywidgets in Jupyter it often updates. 
#         # Better to just change the state and let user see it.
#         # We can implement a timer or just leave it. 
#         # For simplicity, we just print validation.
#         print("Assignments finalized and saved to AnnData object.")
        
#         # Reset button text after a short delay (simulated by just not doing it or requiring another click, 
#         # but let's just leave it as 'Saved!' until next interaction or simple print is enough)
#         # Reverting button style after a moment requires async which is complex here.
#         # Let's just stick to print log which is robust.

#     def show(self):
#         self._fig_widget = self._create_feature_scatter(self.current_feature)
#         self._fig_widget.data[0].on_selection(self._on_selection)
#         self._cluster_fig_widget = self._create_cluster_scatter()

#         self._feature_dropdown = widgets.Dropdown(options=self.features, value=self.current_feature, description='Feature:')
#         self._feature_dropdown.observe(self._on_feature_change)
        
#         self._colormap_dropdown = widgets.Dropdown(options=self.available_colormaps, value=self.current_colormap, description='Colormap:')
#         self._colormap_dropdown.observe(self._on_colormap_change)

#         self._cluster_name_input = widgets.Text(placeholder='Enter cluster name', description='Name:')
#         self._label_button = widgets.Button(description='Label Selection', button_style='success', icon='check')
#         self._label_button.on_click(self._label_selection)
#         self._clear_button = widgets.Button(description='Clear Selection', button_style='warning', icon='times')
#         self._clear_button.on_click(self._clear_selection)
        
#         self._train_button = widgets.Button(description='Train Classifier', button_style='primary', icon='cogs')
#         self._train_button.on_click(self._train_classifier)
        
#         self._threshold_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Threshold:')
#         self._threshold_slider.observe(self._apply_threshold, names='value')

#         self._finalize_button = widgets.Button(description='Finalize Labels', button_style='success', icon='save')
#         self._finalize_button.on_click(self._finalize_assignments)

#         self._rename_cluster_dropdown = widgets.Dropdown(options=[('No clusters', None)], description='Cluster:')
#         self._rename_input = widgets.Text(placeholder='New name', description='New name:')
#         self._rename_button = widgets.Button(description='Rename', button_style='info', icon='edit')
#         self._rename_button.on_click(self._on_rename_cluster)
        
#         self._remove_button = widgets.Button(description='Remove', button_style='danger', icon='trash')
#         self._remove_button.on_click(self._on_remove_cluster)
        
#         self._reorder_button = widgets.Button(description='Reorder IDs', icon='sort-numeric-asc')
#         self._reorder_button.on_click(self._on_reorder_ids)

#         self._summary_text = widgets.Textarea(disabled=True, layout=widgets.Layout(width='100%', height='60px'))
#         self._update_summary()

#         ui = widgets.VBox([
#             widgets.HBox([self._feature_dropdown, self._colormap_dropdown, self._cluster_name_input, self._label_button, self._clear_button]),
#             widgets.HBox([self._train_button, self._threshold_slider, self._finalize_button]),
#             widgets.HBox([self._rename_cluster_dropdown, self._rename_input, self._rename_button, self._remove_button, self._reorder_button]),
#             self._summary_text,
#             widgets.HBox([self._fig_widget, self._cluster_fig_widget])
#         ])
#         display(ui)


# def create_cutoff_interface(df, s=0.1):
#     """
#     Create three interactive scatter plots (IdU vs. pRb, CyclinB1, H3S28p) with histograms and cutoff lines.
#     """
#     N_total = len(df)
#     P = np.array(["N/A"] * N_total, dtype=object)

#     x1_vals = df["pRb"].values
#     x2_vals = df["CyclinB1"].values
#     x3_vals = df["H3S28p"].values
#     y_vals  = df["IdU"].values

#     fig = plt.figure(figsize=(14, 6))
#     gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 4], wspace=0.3, hspace=0.3)
#     fig.subplots_adjust(right=0.75)

#     ax_hist1 = fig.add_subplot(gs[0, 0])
#     ax_hist2 = fig.add_subplot(gs[0, 1])
#     ax_hist3 = fig.add_subplot(gs[0, 2])
#     ax_scatter1 = fig.add_subplot(gs[1, 0])
#     ax_scatter2 = fig.add_subplot(gs[1, 1])
#     ax_scatter3 = fig.add_subplot(gs[1, 2], sharey=ax_scatter1)

#     sns.histplot(x=x1_vals, bins=30, color="gray", alpha=0.7, ax=ax_hist1)
#     sns.histplot(x=x2_vals, bins=30, color="gray", alpha=0.7, ax=ax_hist2)
#     sns.histplot(x=x3_vals, bins=30, color="gray", alpha=0.7, ax=ax_hist3)
#     for ax in (ax_hist1, ax_hist2, ax_hist3): ax.axis('off')

#     sns.scatterplot(x=x1_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter1, color="lightgray", legend=False)
#     ax_scatter1.set_xlabel("pRb"); ax_scatter1.set_ylabel("IdU")

#     sns.scatterplot(x=x2_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter2, color="lightgray", legend=False)
#     ax_scatter2.set_xlabel("CyclinB1")

#     sns.scatterplot(x=x3_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter3, color="lightgray", legend=False)
#     ax_scatter3.set_xlabel("H3S28p")

#     initial_x1, initial_x2, initial_x3, initial_y = map(np.median, [x1_vals, x2_vals, x3_vals, y_vals])

#     vline_hist1 = ax_hist1.axvline(initial_x1, color="red", linewidth=2)
#     vline_hist2 = ax_hist2.axvline(initial_x2, color="red", linewidth=2)
#     vline_hist3 = ax_hist3.axvline(initial_x3, color="red", linewidth=2)
#     vline1 = ax_scatter1.axvline(initial_x1, color="red", linewidth=2)
#     hline1 = ax_scatter1.axhline(initial_y,  color="blue", linewidth=2)
#     vline2 = ax_scatter2.axvline(initial_x2, color="red", linewidth=2)
#     hline2 = ax_scatter2.axhline(initial_y,  color="blue", linewidth=2)
#     vline3 = ax_scatter3.axvline(initial_x3, color="red", linewidth=2)
#     hline3 = ax_scatter3.axhline(initial_y,  color="blue", linewidth=2)

#     plt.show()

#     def make_slider(val, vals, desc):
#         return widgets.FloatSlider(value=val, min=np.min(vals), max=np.max(vals), 
#                                    step=(np.max(vals)-np.min(vals))/200, description=desc, layout=widgets.Layout(width="300px"))

#     slider_x1 = make_slider(initial_x1, x1_vals, "pRb cutoff")
#     slider_y1 = make_slider(initial_y, y_vals, "IdU cutoff")
#     slider_x2 = make_slider(initial_x2, x2_vals, "CyclinB1 cutoff")
#     slider_x3 = make_slider(initial_x3, x3_vals, "H3S28p cutoff")

#     def ReDrawHist():
#         P_local = np.array(["N/A"] * N_total, dtype=object)
#         M0 = df["pRb"].values < slider_x1.value
#         P_local[M0] = "G0"
#         M_s = (df["IdU"].values > slider_y1.value) & (P_local == "N/A")
#         P_local[M_s] = "S"
#         M_g1 = (df["CyclinB1"].values < slider_x2.value) & (P_local == "N/A")
#         P_local[M_g1] = "G1"
#         M_g2 = (df["CyclinB1"].values > slider_x2.value) & (P_local == "N/A")
#         P_local[M_g2] = "G2"
#         M_m = (df["H3S28p"].values > slider_x3.value) & (P_local == "G2")
#         P_local[M_m] = "M"

#         for ax in (ax_scatter1, ax_scatter2, ax_scatter3):
#             for coll in list(ax.collections): coll.remove()

#         sns.scatterplot(x=x1_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter1, color="lightgray", legend=False)
#         sns.scatterplot(x=x2_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter2, color="lightgray", legend=False)
#         sns.scatterplot(x=x3_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter3, color="lightgray", legend=False)

#         colors_map = {"G0": "gray", "S": "red", "G1": "green", "G2": "blue", "M": "magenta"}
#         for ph, color in colors_map.items():
#             mask = (P_local == ph)
#             if mask.any():
#                 sns.scatterplot(x=x1_vals[mask], y=y_vals[mask], s=s, alpha=1, ax=ax_scatter1, color=color, legend=False)
#                 sns.scatterplot(x=x2_vals[mask], y=y_vals[mask], s=s, alpha=1, ax=ax_scatter2, color=color, legend=False)
#                 sns.scatterplot(x=x3_vals[mask], y=y_vals[mask], s=s, alpha=1, ax=ax_scatter3, color=color, legend=False)
#         return P_local

#     def update_lines(change):
#         vline_hist1.set_xdata([slider_x1.value]*2); vline1.set_xdata([slider_x1.value]*2)
#         vline_hist2.set_xdata([slider_x2.value]*2); vline2.set_xdata([slider_x2.value]*2)
#         vline_hist3.set_xdata([slider_x3.value]*2); vline3.set_xdata([slider_x3.value]*2)
#         hline1.set_ydata([slider_y1.value]*2); hline2.set_ydata([slider_y1.value]*2); hline3.set_ydata([slider_y1.value]*2)
#         fig.canvas.draw_idle()

#     slider_x1.observe(update_lines, names="value"); slider_y1.observe(update_lines, names="value")
#     slider_x2.observe(update_lines, names="value"); slider_x3.observe(update_lines, names="value")

#     results = {}
#     button = widgets.Button(description="Get All Cutoffs", button_style="info")
#     out = widgets.Output()

#     def on_button_click(b):
#         with out:
#             clear_output()
#             results.update({
#                 "pRb_cutoff": slider_x1.value, "IdU_cutoff": slider_y1.value,
#                 "CyclinB1_cutoff": slider_x2.value, "H3S28p_cutoff": slider_x3.value
#             })
#             print("Cutoffs:", results)

#     button.on_click(on_button_click)
#     ui = widgets.VBox([widgets.HBox([slider_x1, slider_x2, slider_x3]), widgets.HBox([slider_y1]), widgets.HBox([button, out])])
#     display(ui)
#     return results, P

# def ManualSelection(df, id_column="region_id", x_col="x", y_col="y"):
#     """
#     Interactive manual selection of points using a lasso tool.
#     """
#     df = df.copy()
#     if id_column not in df.columns: df[id_column] = -1
#     label_column = f"{id_column}_Label"
#     if label_column not in df.columns: df[label_column] = ""

#     color_list = ['lightgray'] + list(plt.cm.tab10.colors)
#     cmap = ListedColormap(color_list)
#     norm = BoundaryNorm(boundaries=np.arange(-1.5, len(color_list) - 0.5), ncolors=len(color_list))

#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
#     sc1 = ax1.scatter(df[x_col], df[y_col], s=8, c=[0]*len(df), cmap=cmap, norm=norm)
    
#     valid_cols = [col for col in df.columns if col not in [x_col, y_col, id_column, label_column]]
#     default_val = valid_cols[0] if valid_cols else None
    
#     if default_val:
#         vmin, vmax = np.quantile(df[default_val], 0.01), np.quantile(df[default_val], 0.99)
#         ax2.scatter(df[x_col], df[y_col], s=8, c=df[default_val], cmap='seismic', vmin=vmin, vmax=vmax)
    
#     selected_indices = set()
#     labels_dict = {}
#     selection_counter = {"count": 0}

#     class DualLasso:
#         def __init__(self, ax1, ax2, onselect):
#             self.ax1, self.ax2, self.onselect = ax1, ax2, onselect
#             self.lasso = LassoSelector(ax1, onselect=self._on_select)
#         def _on_select(self, verts): self.onselect(verts)

#     def onselect(verts):
#         path = Path(verts)
#         ind = np.nonzero(path.contains_points(df[[x_col, y_col]].values))[0]
#         selected_indices.clear()
#         selected_indices.update(ind)

#     DualLasso(ax1, ax2, onselect)
#     plt.show()
#     return df


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.widgets import LassoSelector
from matplotlib.path import Path
from matplotlib.patches import Polygon
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from typing import Optional, List, Dict, Tuple
import warnings
import random

class InteractiveClusterLabeler:
    """
    Interactive cluster labeling using Plotly lasso selection on UMAP space.
    """

    def __init__(self, adata, umap_key: str = 'X_umap',
                 features: Optional[List[str]] = None,
                 subsample: Optional[int] = None):
        """
        Initialize the interactive cluster labeler.

        Parameters
        ----------
        adata : AnnData
            Annotated data matrix with UMAP coordinates in obsm
        umap_key : str
            Key in adata.obsm for UMAP coordinates (default: 'X_umap')
        features : list of str, optional
            List of features (markers) to make available for visualization.
            If None, uses all features in adata.var_names
        subsample : int, optional
            If provided, subsample to this many cells for faster interaction
        """
        self.adata_original = adata
        self.umap_key = umap_key

        # Subsample if requested
        if subsample is not None and subsample < adata.n_obs:
            print(f"Subsampling {subsample} cells from {adata.n_obs}")
            self.subsample_indices = np.random.choice(
                adata.n_obs, size=subsample, replace=False
            )
            self.adata = adata[self.subsample_indices].copy()
        else:
            self.adata = adata
            self.subsample_indices = None

        # Extract UMAP coordinates
        if umap_key not in self.adata.obsm:
            raise ValueError(f"UMAP key '{umap_key}' not found in adata.obsm")
        self.umap = self.adata.obsm[umap_key]

        # Set up features
        if features is None:
            self.features = list(self.adata.var_names)
        else:
            # Validate features exist
            missing = [f for f in features if f not in self.adata.var_names]
            if missing:
                raise ValueError(f"Features not found: {missing}")
            self.features = features

        # Initialize cluster labels (-1 = unlabeled)
        self.cluster_labels = -np.ones(len(self.adata), dtype=int)
        self.cluster_names: Dict[int, str] = {}  # cluster_id -> name
        self.next_cluster_id = 0

        # XGBoost model and predictions
        self.classifier = None
        self.predicted_labels = None
        self.predicted_proba = None
        self.prob_threshold = 0.5

        # Available colormaps for feature visualization (Plotly colorscale names)
        self.available_colormaps = [
            'viridis', 'plasma', 'inferno', 'magma', 'cividis',
            'blues', 'greens', 'reds', 'purples', 'oranges',
            'ylorrd', 'ylgnbu', 'turbo', 'hot', 'jet', 'rainbow',
            'rdbu', 'rdylbu', 'rdylgn', 'spectral', 'piyg', 'brbg',
            'puor', 'prgn', 'picnic', 'portland', 'earth',
            'icefire', 'balance', 'curl', 'delta', 'tealrose'
        ]
        self.current_colormap = 'rdbu'

        # UI state
        self.current_feature = self.features[0] if self.features else None
        self._selected_indices = []
        self._fig_widget = None
        self._cluster_fig_widget = None

    def _get_feature_values(self, feature: str) -> np.ndarray:
        """Extract feature values from adata."""
        if feature not in self.adata.var_names:
            raise ValueError(f"Feature '{feature}' not found")

        F = self.adata[:, feature].X
        if hasattr(F, 'toarray'):
            F = F.toarray().flatten()
        else:
            F = np.asarray(F).flatten()
        return F

    def _get_colorbar_range(self, values: np.ndarray) -> Tuple[float, float, float]:
        """
        Get colorbar range with robust percentile scaling, centered at zero.
        """
        p01 = np.nanpercentile(values, 1)
        p99 = np.nanpercentile(values, 99)
        # abs_max = max(abs(p01), abs(p99))
        # vmin, vmax = -abs_max, abs_max
        vmid = 0.0
        vmin = p01
        vmax = p99
        return vmin, vmid, vmax

    def _create_feature_scatter(self, feature: str) -> go.FigureWidget:
        """Create a scatter plot colored by feature values."""
        values = self._get_feature_values(feature)
        vmin, vmid, vmax = self._get_colorbar_range(values)

        fig = go.FigureWidget()
        fig.add_trace(go.Scatter(
            x=self.umap[:, 0],
            y=self.umap[:, 1],
            mode='markers',
            marker=dict(
                size=4,
                color=values,
                colorscale=self.current_colormap,
                cmin=vmin,
                # cmid=vmid,
                cmax=vmax,
                showscale=True,
                colorbar=dict(title=feature, x=1.02),
            ),
            text=[f"Cell {i}<br>{feature}: {values[i]:.2f}"
                  for i in range(len(self.umap))],
            hoverinfo='text',
            name=feature,
            selectedpoints=[],
        ))

        fig.update_layout(
            title=f"UMAP colored by {feature} - Use Lasso to Select",
            xaxis_title="UMAP 1",
            yaxis_title="UMAP 2",
            dragmode='lasso',
            height=500,
            width=600,
            showlegend=False,
        )
        return fig

    def _create_cluster_scatter(self) -> go.FigureWidget:
        """Create a scatter plot colored by cluster assignments with legend."""
        fig = go.FigureWidget()
        colors = px.colors.qualitative.Plotly
        self._add_cluster_traces(fig, colors)
        fig.update_layout(
            title="UMAP colored by Cluster Labels",
            xaxis_title="UMAP 1",
            yaxis_title="UMAP 2",
            height=500,
            width=650,
            showlegend=True,
            legend=dict(
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=1.02,
                bgcolor="rgba(255,255,255,0.8)",
            ),
        )
        return fig

    def _add_cluster_traces(self, fig: go.FigureWidget, colors: list):
        """Add scatter traces for each cluster to the figure."""
        unlabeled_mask = self.cluster_labels == -1
        if unlabeled_mask.any():
            fig.add_trace(go.Scatter(
                x=self.umap[unlabeled_mask, 0],
                y=self.umap[unlabeled_mask, 1],
                mode='markers',
                marker=dict(size=4, color='lightgray', opacity=0.5),
                text=[f"Cell {i}: Unlabeled" for i in np.where(unlabeled_mask)[0]],
                hoverinfo='text',
                name='Unlabeled',
                showlegend=True,
            ))

        for cluster_id in sorted(self.cluster_names.keys()):
            mask = self.cluster_labels == cluster_id
            if not mask.any():
                continue

            color = colors[cluster_id % len(colors)]
            name = self.cluster_names.get(cluster_id, f"Cluster {cluster_id}")

            fig.add_trace(go.Scatter(
                x=self.umap[mask, 0],
                y=self.umap[mask, 1],
                mode='markers',
                marker=dict(size=4, color=color, opacity=0.7),
                text=[f"Cell {i}: {name}" for i in np.where(mask)[0]],
                hoverinfo='text',
                name=name,
                showlegend=True,
            ))

    def _update_cluster_scatter(self):
        """Update the cluster scatter plot with current labels."""
        if self._cluster_fig_widget is None:
            return
        colors = px.colors.qualitative.Plotly
        self._cluster_fig_widget.data = []
        self._add_cluster_traces(self._cluster_fig_widget, colors)

    def _on_selection(self, trace, points, selector):
        """Handle lasso selection events."""
        self._selected_indices = list(points.point_inds)
        n_selected = len(self._selected_indices)
        if n_selected > 0:
            print(f"Selected {n_selected} points")

    def label_points(self, indices: List[int], cluster_name: Optional[str] = None) -> int:
        """Programmatically label a set of points as a new cluster."""
        if not indices:
            print("No indices provided")
            return -1

        if cluster_name is None:
            cluster_name = f"Cluster {self.next_cluster_id}"

        cluster_id = self.next_cluster_id

        indices = [i for i in indices if 0 <= i < len(self.cluster_labels)]
        n_labeled = 0
        for idx in indices:
            if self.cluster_labels[idx] == -1:
                self.cluster_labels[idx] = cluster_id
                n_labeled += 1

        if n_labeled > 0:
            self.cluster_names[cluster_id] = cluster_name
            self.next_cluster_id += 1
            print(f"Labeled {n_labeled} points as '{cluster_name}' (ID: {cluster_id})")
            self._update_cluster_scatter()
            self._update_summary()
            self._update_rename_dropdown()
            return cluster_id
        else:
            print("No new points labeled")
            return -1

    def _label_selection(self, button):
        """Label currently selected points with a new cluster."""
        if not self._selected_indices:
            print("No points selected.")
            return

        cluster_name = self._cluster_name_input.value.strip() or f"Cluster {self.next_cluster_id}"
        
        n_labeled = 0
        cluster_id = self.next_cluster_id
        for idx in self._selected_indices:
            if self.cluster_labels[idx] == -1:
                self.cluster_labels[idx] = cluster_id
                n_labeled += 1
        
        if n_labeled > 0:
            self.cluster_names[cluster_id] = cluster_name
            self.next_cluster_id += 1
            print(f"Labeled {n_labeled} points as '{cluster_name}'")
            self._update_cluster_scatter()
            self._update_summary()
            self._update_rename_dropdown()
            self._selected_indices = []
            self._cluster_name_input.value = ""
        else:
            print("No new points labeled")

    def _clear_selection(self, button):
        self._selected_indices = []
        print("Selection cleared")

    def _on_feature_change(self, change):
        if change['name'] != 'value': return
        feature = change['new']
        self.current_feature = feature
        values = self._get_feature_values(feature)
        vmin, vmid, vmax = self._get_colorbar_range(values)
        with self._fig_widget.batch_update():
            self._fig_widget.data[0].marker.color = values
            self._fig_widget.data[0].marker.cmin = vmin
            self._fig_widget.data[0].marker.cmid = vmid
            self._fig_widget.data[0].marker.cmax = vmax
            self._fig_widget.data[0].marker.colorbar.title = feature
            self._fig_widget.data[0].text = [f"Cell {i}<br>{feature}: {values[i]:.2f}" for i in range(len(self.umap))]
            self._fig_widget.layout.title = f"UMAP colored by {feature} - Use Lasso to Select"

    def _on_colormap_change(self, change):
        if change['name'] != 'value': return
        colormap = change['new']
        self.current_colormap = colormap
        if self._fig_widget is not None:
            with self._fig_widget.batch_update():
                self._fig_widget.data[0].marker.colorscale = colormap

    def _on_rename_cluster(self, button):
        cluster_id = self._rename_cluster_dropdown.value
        new_name = self._rename_input.value.strip()
        if cluster_id is None or not new_name: return
        self.cluster_names[cluster_id] = new_name
        self._rename_input.value = ""
        self._update_cluster_scatter()
        self._update_summary()
        self._update_rename_dropdown()

    def _update_rename_dropdown(self):
        if not hasattr(self, '_rename_cluster_dropdown'): return
        options = [(f"{self.cluster_names.get(cid, f'Cluster {cid}')} (ID: {cid})", cid)
                   for cid in sorted(self.cluster_names.keys())]
        self._rename_cluster_dropdown.options = options if options else [('No clusters', None)]

    def _on_remove_cluster(self, button):
        cluster_id = self._rename_cluster_dropdown.value
        if cluster_id is None: return
        self.cluster_labels[self.cluster_labels == cluster_id] = -1
        del self.cluster_names[cluster_id]
        self._update_cluster_scatter()
        self._update_summary()
        self._update_rename_dropdown()

    def _on_reorder_ids(self, button):
        if not self.cluster_names: return
        old_ids = sorted(self.cluster_names.keys())
        id_mapping = {old_id: new_id for new_id, old_id in enumerate(old_ids)}
        new_labels = self.cluster_labels.copy()
        for old_id, new_id in id_mapping.items():
            new_labels[self.cluster_labels == old_id] = new_id
        self.cluster_labels = new_labels
        self.cluster_names = {id_mapping[old_id]: name for old_id, name in self.cluster_names.items()}
        self.next_cluster_id = len(self.cluster_names)
        self._update_cluster_scatter()
        self._update_summary()
        self._update_rename_dropdown()

    def _train_classifier(self, button):
        try:
            import xgboost as xgb
        except ImportError:
            print("XGBoost not installed. Run: pip install xgboost")
            return

        labeled_mask = self.cluster_labels >= 0
        if labeled_mask.sum() == 0:
            print("No points labeled yet.")
            return

        unique_labels = np.unique(self.cluster_labels[labeled_mask])
        if len(unique_labels) < 2:
            print("Need at least 2 different clusters.")
            return

        print(f"Training XGBoost classifier on {labeled_mask.sum()} labeled points...")
        X = self.adata.X
        if hasattr(X, 'toarray'): X = X.toarray()
        X_train = X[labeled_mask]
        y_train = self.cluster_labels[labeled_mask]

        self.classifier = xgb.XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=len(unique_labels),
            random_state=42, verbosity=0,
        )

        label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        idx_to_label = {idx: label for label, idx in label_to_idx.items()}
        y_train_mapped = np.array([label_to_idx[y] for y in y_train])

        self.classifier.fit(X_train, y_train_mapped)
        
        # Predict on the FULL dataset
        X_full = self.adata_original.X
        if hasattr(X_full, 'toarray'): X_full = X_full.toarray()
        
        proba_full = self.classifier.predict_proba(X_full)
        pred_idx_full = np.argmax(proba_full, axis=1)
        
        self.full_predicted_labels = np.array([idx_to_label[idx] for idx in pred_idx_full])
        self.full_predicted_proba = np.max(proba_full, axis=1)

        # If subsampled, map back to subsample for UI
        if self.subsample_indices is not None:
            self.predicted_labels = self.full_predicted_labels[self.subsample_indices]
            self.predicted_proba = self.full_predicted_proba[self.subsample_indices]
        else:
            self.predicted_labels = self.full_predicted_labels
            self.predicted_proba = self.full_predicted_proba

        print(f"Classifier trained successfully! Predictions generated for {len(self.adata_original)} cells.")
        self._apply_threshold(None)

    def _apply_threshold(self, change):
        if self.predicted_labels is None: return
        threshold = self._threshold_slider.value if hasattr(self, '_threshold_slider') else self.prob_threshold
        self.prob_threshold = threshold

        final_labels = self.cluster_labels.copy()
        unlabeled_mask = self.cluster_labels == -1
        confident_mask = self.predicted_proba >= threshold
        
        predicted_confident = unlabeled_mask & confident_mask
        final_labels[predicted_confident] = self.predicted_labels[predicted_confident]
        
        predicted_uncertain = unlabeled_mask & ~confident_mask
        final_labels[predicted_uncertain] = -2
        
        self._final_labels = final_labels
        self._update_final_cluster_plot()

    def _update_final_cluster_plot(self):
        if self._cluster_fig_widget is None or not hasattr(self, '_final_labels'): return
        colors = px.colors.qualitative.Plotly
        self._cluster_fig_widget.data = []

        na_mask = self._final_labels == -2
        if na_mask.any():
            self._cluster_fig_widget.add_trace(go.Scatter(
                x=self.umap[na_mask, 0], y=self.umap[na_mask, 1],
                mode='markers', marker=dict(size=4, color='black', opacity=0.7),
                text=[f"Cell {i}: NA (prob={self.predicted_proba[i]:.2f})" for i in np.where(na_mask)[0]],
                hoverinfo='text', name='NA (uncertain)', showlegend=True,
            ))
            
        unlabeled_mask = self._final_labels == -1
        if unlabeled_mask.any():
            self._cluster_fig_widget.add_trace(go.Scatter(
                x=self.umap[unlabeled_mask, 0], y=self.umap[unlabeled_mask, 1],
                mode='markers', marker=dict(size=4, color='lightgray', opacity=0.5),
                name='Unlabeled', showlegend=True,
            ))

        for cluster_id in sorted(self.cluster_names.keys()):
            mask = self._final_labels == cluster_id
            if not mask.any(): continue
            color = colors[cluster_id % len(colors)]
            name = self.cluster_names.get(cluster_id, f"Cluster {cluster_id}")
            hover_texts = []
            for i in np.where(mask)[0]:
                if self.cluster_labels[i] >= 0:
                    hover_texts.append(f"Cell {i}: {name} (manual)")
                else:
                    hover_texts.append(f"Cell {i}: {name} (pred, p={self.predicted_proba[i]:.2f})")
            
            self._cluster_fig_widget.add_trace(go.Scatter(
                x=self.umap[mask, 0], y=self.umap[mask, 1],
                mode='markers', marker=dict(size=4, color=color, opacity=0.7),
                text=hover_texts, hoverinfo='text', name=name, showlegend=True,
            ))

    def save_full_labels(self, obs_key: str = 'predicted_labels', prob_key: str = 'predicted_proba'):
        """
        Save the predicted labels and probabilities for the FULL dataset to adata_original.obs.
        
        Parameters
        ----------
        obs_key : str
            Key to store the predicted labels in adata_original.obs
        prob_key : str
            Key to store the predicted probabilities in adata_original.obs
        """
        if not hasattr(self, 'full_predicted_labels'):
            print("No classifier trained yet. Train a classifier first.")
            return

        # Apply threshold to full predictions
        final_full_labels = self.full_predicted_labels.copy().astype(object)
        uncertain_mask = self.full_predicted_proba < self.prob_threshold
        final_full_labels[uncertain_mask] = 'Uncertain' # Or keep as string

        self.adata_original.obs[obs_key] = final_full_labels
        self.adata_original.obs[prob_key] = self.full_predicted_proba
        print(f"Saved full predictions to adata.obs['{obs_key}'] and adata.obs['{prob_key}']")


    def _update_summary(self):
        n_labeled = (self.cluster_labels >= 0).sum()
        n_unlabeled = (self.cluster_labels == -1).sum()
        n_clusters = len(self.cluster_names)
        summary = f"Labeled: {n_labeled} | Unlabeled: {n_unlabeled} | Clusters: {n_clusters}"
        if hasattr(self, '_summary_text'):
            self._summary_text.value = summary

    def _finalize_assignments(self, button):
        """Finalize the current assignments based on threshold and save to adata."""
        if not hasattr(self, 'full_predicted_labels'):
            print("No classifier trained yet.")
            return
            
        self.save_full_labels()
        self._update_summary() # Update summary to potentially reflect saved state if we decide to track that
        # Provide visual feedback
        button.description = "Saved!"
        button.icon = "check-circle" 
        import time
        # Note: In a real async UI this simple sleep might block, 
        # but for ipywidgets in Jupyter it often updates. 
        # Better to just change the state and let user see it.
        # We can implement a timer or just leave it. 
        # For simplicity, we just print validation.
        print("Assignments finalized and saved to AnnData object.")
        
        # Reset button text after a short delay (simulated by just not doing it or requiring another click, 
        # but let's just leave it as 'Saved!' until next interaction or simple print is enough)
        # Reverting button style after a moment requires async which is complex here.
        # Let's just stick to print log which is robust.

    def show(self):
        self._fig_widget = self._create_feature_scatter(self.current_feature)
        self._fig_widget.data[0].on_selection(self._on_selection)
        self._cluster_fig_widget = self._create_cluster_scatter()

        self._feature_dropdown = widgets.Dropdown(options=self.features, value=self.current_feature, description='Feature:')
        self._feature_dropdown.observe(self._on_feature_change)
        
        self._colormap_dropdown = widgets.Dropdown(options=self.available_colormaps, value=self.current_colormap, description='Colormap:')
        self._colormap_dropdown.observe(self._on_colormap_change)

        self._cluster_name_input = widgets.Text(placeholder='Enter cluster name', description='Name:')
        self._label_button = widgets.Button(description='Label Selection', button_style='success', icon='check')
        self._label_button.on_click(self._label_selection)
        self._clear_button = widgets.Button(description='Clear Selection', button_style='warning', icon='times')
        self._clear_button.on_click(self._clear_selection)
        
        self._train_button = widgets.Button(description='Train Classifier', button_style='primary', icon='cogs')
        self._train_button.on_click(self._train_classifier)
        
        self._threshold_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Threshold:')
        self._threshold_slider.observe(self._apply_threshold, names='value')

        self._finalize_button = widgets.Button(description='Finalize Labels', button_style='success', icon='save')
        self._finalize_button.on_click(self._finalize_assignments)

        self._rename_cluster_dropdown = widgets.Dropdown(options=[('No clusters', None)], description='Cluster:')
        self._rename_input = widgets.Text(placeholder='New name', description='New name:')
        self._rename_button = widgets.Button(description='Rename', button_style='info', icon='edit')
        self._rename_button.on_click(self._on_rename_cluster)
        
        self._remove_button = widgets.Button(description='Remove', button_style='danger', icon='trash')
        self._remove_button.on_click(self._on_remove_cluster)
        
        self._reorder_button = widgets.Button(description='Reorder IDs', icon='sort-numeric-asc')
        self._reorder_button.on_click(self._on_reorder_ids)

        self._summary_text = widgets.Textarea(disabled=True, layout=widgets.Layout(width='100%', height='60px'))
        self._update_summary()

        ui = widgets.VBox([
            widgets.HBox([self._feature_dropdown, self._colormap_dropdown, self._cluster_name_input, self._label_button, self._clear_button]),
            widgets.HBox([self._train_button, self._threshold_slider, self._finalize_button]),
            widgets.HBox([self._rename_cluster_dropdown, self._rename_input, self._rename_button, self._remove_button, self._reorder_button]),
            self._summary_text,
            widgets.HBox([self._fig_widget, self._cluster_fig_widget])
        ])
        display(ui)


def create_cutoff_interface(df, s=0.1):
    """
    Create three interactive scatter plots (IdU vs. pRb, CyclinB1, H3S28p) with histograms and cutoff lines.
    """
    N_total = len(df)
    P = np.array(["N/A"] * N_total, dtype=object)

    x1_vals = df["pRb"].values
    x2_vals = df["CyclinB1"].values
    x3_vals = df["H3S28p"].values
    y_vals  = df["IdU"].values

    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 4], wspace=0.3, hspace=0.3)
    fig.subplots_adjust(right=0.75)

    ax_hist1 = fig.add_subplot(gs[0, 0])
    ax_hist2 = fig.add_subplot(gs[0, 1])
    ax_hist3 = fig.add_subplot(gs[0, 2])
    ax_scatter1 = fig.add_subplot(gs[1, 0])
    ax_scatter2 = fig.add_subplot(gs[1, 1])
    ax_scatter3 = fig.add_subplot(gs[1, 2], sharey=ax_scatter1)

    sns.histplot(x=x1_vals, bins=30, color="gray", alpha=0.7, ax=ax_hist1)
    sns.histplot(x=x2_vals, bins=30, color="gray", alpha=0.7, ax=ax_hist2)
    sns.histplot(x=x3_vals, bins=30, color="gray", alpha=0.7, ax=ax_hist3)
    for ax in (ax_hist1, ax_hist2, ax_hist3): ax.axis('off')

    sns.scatterplot(x=x1_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter1, color="lightgray", legend=False)
    ax_scatter1.set_xlabel("pRb"); ax_scatter1.set_ylabel("IdU")

    sns.scatterplot(x=x2_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter2, color="lightgray", legend=False)
    ax_scatter2.set_xlabel("CyclinB1")

    sns.scatterplot(x=x3_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter3, color="lightgray", legend=False)
    ax_scatter3.set_xlabel("H3S28p")

    initial_x1, initial_x2, initial_x3, initial_y = map(np.median, [x1_vals, x2_vals, x3_vals, y_vals])

    vline_hist1 = ax_hist1.axvline(initial_x1, color="red", linewidth=2)
    vline_hist2 = ax_hist2.axvline(initial_x2, color="red", linewidth=2)
    vline_hist3 = ax_hist3.axvline(initial_x3, color="red", linewidth=2)
    vline1 = ax_scatter1.axvline(initial_x1, color="red", linewidth=2)
    hline1 = ax_scatter1.axhline(initial_y,  color="blue", linewidth=2)
    vline2 = ax_scatter2.axvline(initial_x2, color="red", linewidth=2)
    hline2 = ax_scatter2.axhline(initial_y,  color="blue", linewidth=2)
    vline3 = ax_scatter3.axvline(initial_x3, color="red", linewidth=2)
    hline3 = ax_scatter3.axhline(initial_y,  color="blue", linewidth=2)

    plt.show()

    def make_slider(val, vals, desc):
        return widgets.FloatSlider(value=val, min=np.min(vals), max=np.max(vals), 
                                   step=(np.max(vals)-np.min(vals))/200, description=desc, layout=widgets.Layout(width="300px"))

    slider_x1 = make_slider(initial_x1, x1_vals, "pRb cutoff")
    slider_y1 = make_slider(initial_y, y_vals, "IdU cutoff")
    slider_x2 = make_slider(initial_x2, x2_vals, "CyclinB1 cutoff")
    slider_x3 = make_slider(initial_x3, x3_vals, "H3S28p cutoff")

    def ReDrawHist():
        P_local = np.array(["N/A"] * N_total, dtype=object)
        M0 = df["pRb"].values < slider_x1.value
        P_local[M0] = "G0"
        M_s = (df["IdU"].values > slider_y1.value) & (P_local == "N/A")
        P_local[M_s] = "S"
        M_g1 = (df["CyclinB1"].values < slider_x2.value) & (P_local == "N/A")
        P_local[M_g1] = "G1"
        M_g2 = (df["CyclinB1"].values > slider_x2.value) & (P_local == "N/A")
        P_local[M_g2] = "G2"
        M_m = (df["H3S28p"].values > slider_x3.value) & (P_local == "G2")
        P_local[M_m] = "M"

        for ax in (ax_scatter1, ax_scatter2, ax_scatter3):
            for coll in list(ax.collections): coll.remove()

        sns.scatterplot(x=x1_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter1, color="lightgray", legend=False)
        sns.scatterplot(x=x2_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter2, color="lightgray", legend=False)
        sns.scatterplot(x=x3_vals, y=y_vals, s=s, alpha=1, ax=ax_scatter3, color="lightgray", legend=False)

        colors_map = {"G0": "gray", "S": "red", "G1": "green", "G2": "blue", "M": "magenta"}
        for ph, color in colors_map.items():
            mask = (P_local == ph)
            if mask.any():
                sns.scatterplot(x=x1_vals[mask], y=y_vals[mask], s=s, alpha=1, ax=ax_scatter1, color=color, legend=False)
                sns.scatterplot(x=x2_vals[mask], y=y_vals[mask], s=s, alpha=1, ax=ax_scatter2, color=color, legend=False)
                sns.scatterplot(x=x3_vals[mask], y=y_vals[mask], s=s, alpha=1, ax=ax_scatter3, color=color, legend=False)
        return P_local

    def update_lines(change):
        vline_hist1.set_xdata([slider_x1.value]*2); vline1.set_xdata([slider_x1.value]*2)
        vline_hist2.set_xdata([slider_x2.value]*2); vline2.set_xdata([slider_x2.value]*2)
        vline_hist3.set_xdata([slider_x3.value]*2); vline3.set_xdata([slider_x3.value]*2)
        hline1.set_ydata([slider_y1.value]*2); hline2.set_ydata([slider_y1.value]*2); hline3.set_ydata([slider_y1.value]*2)
        fig.canvas.draw_idle()

    slider_x1.observe(update_lines, names="value"); slider_y1.observe(update_lines, names="value")
    slider_x2.observe(update_lines, names="value"); slider_x3.observe(update_lines, names="value")

    results = {}
    button = widgets.Button(description="Get All Cutoffs", button_style="info")
    out = widgets.Output()

    def on_button_click(b):
        with out:
            clear_output()
            results.update({
                "pRb_cutoff": slider_x1.value, "IdU_cutoff": slider_y1.value,
                "CyclinB1_cutoff": slider_x2.value, "H3S28p_cutoff": slider_x3.value
            })
            print("Cutoffs:", results)

    button.on_click(on_button_click)
    ui = widgets.VBox([widgets.HBox([slider_x1, slider_x2, slider_x3]), widgets.HBox([slider_y1]), widgets.HBox([button, out])])
    display(ui)
    return results, P

def ManualSelection(df, id_column="region_id", x_col="x", y_col="y"):
    """
    Interactive manual selection of points using a lasso tool.
    """
    df = df.copy()
    if id_column not in df.columns: df[id_column] = -1
    label_column = f"{id_column}_Label"
    if label_column not in df.columns: df[label_column] = ""

    color_list = ['lightgray'] + list(plt.cm.tab10.colors)
    cmap = ListedColormap(color_list)
    norm = BoundaryNorm(boundaries=np.arange(-1.5, len(color_list) - 0.5), ncolors=len(color_list))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
    sc1 = ax1.scatter(df[x_col], df[y_col], s=8, c=[0]*len(df), cmap=cmap, norm=norm)
    
    valid_cols = [col for col in df.columns if col not in [x_col, y_col, id_column, label_column]]
    default_val = valid_cols[0] if valid_cols else None
    
    if default_val:
        vmin, vmax = np.quantile(df[default_val], 0.01), np.quantile(df[default_val], 0.99)
        ax2.scatter(df[x_col], df[y_col], s=8, c=df[default_val], cmap='seismic', vmin=vmin, vmax=vmax)
    
    selected_indices = set()
    labels_dict = {}
    selection_counter = {"count": 0}

    class DualLasso:
        def __init__(self, ax1, ax2, onselect):
            self.ax1, self.ax2, self.onselect = ax1, ax2, onselect
            self.lasso = LassoSelector(ax1, onselect=self._on_select)
        def _on_select(self, verts): self.onselect(verts)

    def onselect(verts):
        path = Path(verts)
        ind = np.nonzero(path.contains_points(df[[x_col, y_col]].values))[0]
        selected_indices.clear()
        selected_indices.update(ind)

    DualLasso(ax1, ax2, onselect)
    plt.show()
    return df


load normalizes & preprocessed data

In [ ]:
import pandas as pd
add = '/Users/yishai/Dropbox/CyTOF_Breast/data_pdx/temporary_plots/df_b3.parquet'
dir_plots = '/Users/yishai/Dropbox/CyTOF_Breast/data_pdx/temporary_plots/sample_b3__08022026/'
df = pd.read_parquet(add)
for i in df['samp'].unique():
    print(i, len(df[df['samp']==i]))

cols = ['ECad',
 'panKeratin',
 'K5',
 'EpCam',
 'H3K27me2',
 'gH2AX',
 'aSMA',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2Aub',
 'Vimentin',
 'H3K64ac',
 'BMI-1',
 'H3K27ac',
 'H4K20me3',
 'ER',
 'CD49f',
 'CD24',
 'GATA3',
 'H3K9ac',
 'H3K9me3',
 'CD44',
 'Ki67',
 'K8-18',
 'H3K36me3',
 'H3K4me3',
 'H3K27me3',
 'MBD',
 'H3S28p',
 'H4',
 'H3',
 'H3.3']
# df = df[cols]

[col for col in df.columns if col not in cols ]


scale

In [ ]:
from sklearn.preprocessing import StandardScaler
SS=StandardScaler()

# cols = config['groups']['EpiCols'] + config['groups']['Core']
df[cols]=SS.fit_transform(df[cols])
print ('data is scaled using StandardScaler on features:',cols) 
print(cols)
del SS

# df.dropna(inplace=True)
df.reset_index(drop=True,inplace=True)
print(len(df))
df.columns

# samples = Series(df['samp'])
# uniq_samples = samples.unique()# don't drop anchor samples

# uniq_samples

In [ ]:
# import numpy as np
# import pandas as pd
# import scanpy as sc
# import anndata as ad
# # df: cells x features (already normalized / scaled)
# adata = ad.AnnData(X=df.values.copy())
# adata.var_names = df.columns.copy()
# adata.obs_names = df.index.copy()
# sc.settings.verbosity = 3

# sc.pp.neighbors(
#     adata,
#     n_neighbors=10,
#     use_rep='X'   # <-- this is the key line
# )
# sc.tl.umap(adata, min_dist=0.1,verbose=True,n_epochs=200,)

# # sc.tl.umap(adata)


In [ ]:
from umap import UMAP



umap_cols =  ['K5', 'EpCam', 'aSMA', 'Vimentin', 'ER', 'CD49f', 'CD24', 'GATA3', 'CD44', 'K8-18', 'Ki67', 'ECad', 'panKeratin', 'H3K27me2', 'H3K36me2', 'H3K4me1', 'H3K9me2', 'H4K16ac', 'H2Aub', 'H3K64ac', 'H3K27ac', 'H4K20me3', 'BMI-1', 'gH2AX', 'H3K9ac', 'H3K9me3', 'H3S28p', 'H3K36me3', 'H3K4me3', 'H3K27me3', 'MBD']
#
print([col for col in umap_cols if col not in df.columns])
print([col for col in df.columns if col not in umap_cols])

umapData = UMAP(  verbose=True,
                n_neighbors=10,
                min_dist=0.1,
                # n_components=2, metric='euclidean', random_state=42,  densmap=False,
                n_epochs=200,
                # random_state=42,

                ).fit_transform(df[umap_cols].copy(),)
umapData = pd.DataFrame(umapData, columns=['umap1', 'umap2'],index = df.index) 


In [ ]:
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
from matplotlib.pyplot import show,close


def umap_plot(umapData,intensity=None,ind:list[int] = [],title:str = '',figname :str= '',backgroundColor :str= 'gainsboro') -> None:#limits = [None,None,None,None]
    '''
    plot umap with intensity according to specific feature values in each point in map
    umapData - umap coordinates of each point in sample
    intensity - feature values of each point in sample (same order as umapData)
    
    '''

    fig,axs = plt.subplots(1,figsize = (10, 10))

    axs.set_ylabel('umap2')
    axs.set_xlabel('umap1')
    axs.set_title(title)  
    
    # plot settings
    axs.set_facecolor(backgroundColor)
    # if not intensity:
    #     intensity = umapData.copy().mean(axis=1)
    # vmax=2.0;vmin=-1.0
    vmax=intensity.quantile(0.99);vmin=intensity.quantile(0.01)
    
    # colorbar settings
    norm = Normalize(vmax=vmax,vmin=vmin)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm,cmap=plt.cm.seismic),ax  = axs)
    
    
    ind = list(intensity.index) if not ind else ind# if ind is empty take all indexes
    axs.scatter(umapData['umap1'].loc[ind],umapData['umap2'].loc[ind],c=intensity.loc[ind],
                s=2, cmap=plt.cm.seismic, vmax=vmax,vmin=vmin)
    # fig.savefig(dir_plots+figname+title+'.pdf', format='pdf', bbox_inches="tight", pad_inches=0.2)
    # print(dir_plots+figname+title+'.pdf')
    # show()


    # self.figSettings(fig,figname)

In [ ]:
# # import umap

# # umap_model = umap.UMAP(
# #     n_neighbors=15,
# #     min_dist=0.3,
# #     random_state=42
# # )

# # X_umap = umap_model.fit_transform(df.values)





# # %run umap.ipynb   
# from app.functions.clustering import *
# # from clustering import *
# clustering = Clustering(**config)


# from app.functions.plots import *
# plot = Plots(**config)#build Umap_dbscan class (either from MASTER or from here) using parent class containg the config data

# # config

# clustering.features = cols

# umapData = clustering.umap(df.copy(),
#                            columns= None,
#                         #    params = [0.3,30]
#                            # params = [0.1,10]
#                            params = [0.5,15]
#                            )
# umapData = clustering.umap(df)
umap_plot(umapData,df['H4'].copy(),
            title = f'UMAP H4 ',
            # figname = '1_'+config["figname"]+'UMAP_(core)'
            )

for i in ['CD45', 'MHC', 'CD298']:
   umap_plot(umapData,df[i].copy(),
               title = f'UMAP {i} ',
               # figname = '1_'+config["figname"]+'UMAP_(core)'
               )





In [ ]:
# umapData[]
# plot.umap(umapData,df['H4'].copy(),
#             title = f'{config["title"]} UMAP ',
#             figname = '1_'+config["figname"]+'UMAP_(core)')


df1 = df.copy()
umapData1 = umapData.copy()
# samples1 = samples.copy()

# # umapData1.columns
ind1 = umapData1[umapData1['umap1']<7.5].index #NOT STROMA
# ind2 = [i for i in umapData1.index if i not in ind1 ]
ind = ind1


df = df1.loc[ind]
umapData = umapData1.loc[ind]
# samples = samples1.loc[ind]
# clustering.umapData = umapData
df.drop(['CD45', 'MHC', 'CD298'],inplace = True,axis = 1)

umap_plot(umapData,df['H4'].copy(),
            title = f'UMAP without stroma ',
            # figname = '1_'+config["figname"]+'UMAP_(core)'
            )

In [ ]:
adata = ad.AnnData(
    X=df.values.astype(np.float32),
    obs=pd.DataFrame(index=df.index),
    var=pd.DataFrame(index=df.columns),
)
X_umap = umapData.values

adata.obsm["X_umap"] = X_umap


In [ ]:
marker_sets = {
 # Luminal/epithelial program: epithelial & ER axis up; mesenchymal/basal down
 "Epithelial_Luminal": {
 "up": {"EpCam", "ECad", "ER", "GATA3", "K8-18"},
 "down": {"K5", "Vimentin", "aSMA"}
 },
 # “Basal_Noa” (histone-flavor): active H3K4 marks up; repressive H3K9me2 down
 "Basal_Noa": {
 "up": {"H3K4me1", "H3K4me3","H3K9me2"},
 "down": {"H4K20me3","H3K36me3"}
 },


 # Proliferation / cell-cycle & immediate-early signaling up
 "Proliferation": {
 "up": {"Ki67", "H3S28p", "H3K9ac", "H3K64ac"},
 "down": set()
 },
}




verify all needed cols are with same name as in marker sets

In [ ]:
cols_in_marker_sets = ["EpCam", "ECad", "ER", "GATA3", "K8-18","K5", "Vimentin", "aSMA","H3K4me1", "H3K4me3","H3K9me2","H4K20me3","H3K36me3","Ki67", "H3S28p", "H3K9ac", "H3K64ac"
]
missing_cols = [col for col in cols_in_marker_sets if col not in df.columns]
print(f'missing_cols in df: {missing_cols}' if missing_cols else 'no missing cols in df')

# df.columns
# ['ECad', 'panKeratin', 'K5', 'EpCam', 'H3K27me2', 'gH2AX', 'aSMA', 'H3K36me2', 'H3K4me1', 'H3K9me2', 'H4K16ac', 'H2Aub', 'Vimentin', 'H3K64ac', 'BMI-1', 'H3K27ac',
#        'H4K20me3', 'ER', 'CD49f', 'CD24', 'GATA3', 'H3K9ac', 'H3K9me3', 'CD44', 'Ki67', 'K8-18', 'H3K36me3', 'H3K4me3', 'H3K27me3', 'MBD', 'H3S28p', 'samp', 'ind', 'H4', 'H3',
#        'H3.3'],

In [ ]:
results = perm_cell(
    adata.copy(),
    marker_sets
)
# Z_df, P_df, Zdir_df = results
res = results[0]

In [ ]:
import random
# Z_df, P_df, Zdir_df = results
# for i in range(5):

    # res = results[i]
    # type(results)
    # 
    # adata.obs["luminal_score"] = Zdir_df["luminal"]

# cols = ['Epithelial_Luminal', 'Basal_Noa',"Proliferation"]
cols = ['Epithelial_Luminal', 'Basal_Noa',"Proliferation"]

res.index = df.index
df.loc[:, cols] = res.loc[:, cols]

for col in cols:
    umap_plot(umapData,df[col].copy(),
                title = f'{col} - UMAP ',
                # figname = '1_'+config["figname"]+'UMAP_(core)_without_stroma'
                )

# for i in [1]:#[10,50,1]:

#         ind  = random.sample(samples.index.tolist(),len(samples)//i) 
#         plot.umap_by_feature(umapData.copy(),df.copy().loc[ind], cols, #df.copy().loc[ind], config,#limits = limits,
#         title='UMAP ' + config["title"],
#         figname=f'2_{i}_'+config["figname"]+'UMAP_',
#         )

# add the new features to adata df
adata = ad.AnnData(
    X=df.values.astype(np.float32),
    obs=pd.DataFrame(index=df.index),
    var=pd.DataFrame(index=df.columns),
)

adata.obsm["X_umap"] = X_umap

In [ ]:



# import scanpy as sc
# import pandas as pd
# def main(adata):
#     """
#     Entry point for interactive analysis.
#     Choose which interface to run.
#     """

#     # === Example data loading ===
#     # Replace with your real data


#     # Option A: AnnData-based clustering
#     # adata = load_anndata_example()

#     # Option B: Cell-cycle cutoff interface
#     # df_cutoff = load_cutoff_dataframe()

#     # Option C: Manual lasso selection on 2D points
#     # df_manual = load_manual_dataframe()

#     # === Choose what to run ===
#     run_interactive_cluster_labeler(adata)
#     # run_cutoff_interface(df_cutoff)
#     # run_manual_selection(df_manual)

# def run_interactive_cluster_labeler(adata):
#     """
#     Launch the InteractiveClusterLabeler UI.
#     """
#     labeler = InteractiveClusterLabeler(
#         adata=adata,
#         umap_key="X_umap",   # must exist in adata.obsm
#         features=None,      # or pass a list of genes
#         subsample=5000      # or e.g. 5000 for speed
#     )
#     labeler.show()


# main(adata)


In [ ]:
import scanpy as sc
import pandas as pd

def main(adata, subsample = None, features = None) -> InteractiveClusterLabeler:
    """
    Entry point for interactive analysis.
    Launches the interactive cluster labeler and returns the labeler object
    so you can access cluster labels after manual labeling.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data with UMAP coordinates in adata.obsm['X_umap']
    subsample : int
        Number of cells to subsample for interactive labeling

    Returns
    -------
    labeler : InteractiveClusterLabeler
        The labeler object. Access labels via `labeler.cluster_labels` or
        `labeler._final_labels` after classifier/prediction.
    """

    labeler = InteractiveClusterLabeler(
        adata=adata,
        umap_key="X_umap",   # make sure UMAP is in adata.obsm
        features=features,       # or provide a list of genes/features
        subsample=10000
    )

    labeler.show()

  
    return labeler






# === Usage example ===
# Launch the interactive UI
labeler = main(adata, features=cols)

# ['Epithelial_Luminal', 'Basal_Noa',"Proliferation"]



In [ ]:

labeler.save_full_labels(obs_key="predicted_labels", prob_key="predicted_proba")
# adata.obs['predicted_labels']
adata.obs
cols = ['predicted_labels','predicted_proba']

df[cols] = adata.obs[cols]
df.columns
# df = adata.obs['predicted_labels']
# df['predicted_proba'] = adata.obs['predicted_proba']
# # import random
# # Z_df, P_df, Zdir_df = results
# # P_df
# # # type(results)
# # # Z_df, P_df, Zdir_df
# # # adata.obs["luminal_score"] = Zdir_df["luminal"]

# # 
# adata.obs.index = df.index
# df.loc[:, cols] = adata.obs.loc[:, cols]

df.to_parquet(f'{config["dir_plots"]}/df.parquet')
umapData.to_parquet(f'{config["dir_plots"]}/umapData.parquet')

In [ ]:
# cols = ['Epithelial_Luminal', 'Basal_Noa',"Proliferation",'predicted_labels','predicted_proba']

# for i in [1]:#[10,50,1]:

#         ind  = random.sample(samples.index.tolist(),len(samples)//i) 
#         plot.umap_by_feature(umapData.copy(),df.copy().loc[ind], cols, #df.copy().loc[ind], config,#limits = limits,
#         title='UMAP ' + config["title"],
#         figname=f'2_{i}_'+config["figname"]+'UMAP_',
#         )

dbscan

In [ ]:
# c = cluster_colors()
# clustering.umapData
# labels = clustering.DBscan(c,
#                            params = [0.06, 50]
#                            )
# # from plot_functions import drawDbscan


# # len(labels),len(df),len(umapData)
# # print (len(df),len(labels))

# df['labels'] = labels
# plot.DBscan(df['labels'],clustering.dbData,clustering.colors,
#             title=f'{config["title"]} DBscan ',
#             figname='1_'+config["figname"]+'DBscan')



In [ ]:
# from app.functions.heatmap import *


In [ ]:
# from app.functions.heatmap import *

# heatmaps = Heatmap(**config)#build class (either from MASTER or from here) using parent class containg the config data
# df['Cytokeratin-5'] = df['K5'].copy()
# for features_group in heatmaps.features_groups:
#     x = config['groups'][features_group]
#     config['groups'][features_group] = [i if i!= 'K5' else 'Cytokeratin-5' for i in x ]
#     print(features_group,config['groups'][features_group])
# from app.functions.heatmap import *

# heatmaps = Heatmap(**config)#build class (either from MASTER or from here) using parent class containg the config data


In [ ]:


# for features_group in heatmaps.features_groups:# CellCycle features are present only in the '.2' data other wise only epigenetic and cellcycle features
#     heatmaps.plot(  df.copy(), labels.copy(),
#                     features = config['groups'][features_group].copy(),
#                     # ind = list(df.index[labels != -1]),#remove unclustered -1
#                     title='HeatMap clusters -' + config['title'] + ': '+features_group,
#                     figname='3_HeatMap_clusters_'+config['figname']+'using_'+features_group,)
    
        
        
#     print('HeatMap by clusters - done')   


In [ ]:
df.columns